# Model Training

This code loads the extracted feature dataframe and trains classification models on it.
It also evaluates the models.

In [37]:
import numpy as np
import pandas as pd
import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from sklearn.model_selection import GridSearchCV

In [38]:
# importing feature_df
feature_df = pd.read_csv('feature_df.csv')

In [39]:
feature_df.shape

(11550, 49)

In [40]:
feature_df.head()

,dissimilarity,correlation,homogeneity,contrast,ASM,energy,mean_r,mean_g,mean_b,std_r,...,Bin_21,Bin_22,Bin_23,Bin_24,Bin_25,Bin_26,Bin_27,Bin_28,Bin_29,class
0,18.923771,0.759237,0.085817,785.858455,0.000174,0.013178,4.656550,4.672133,4.669419,2.405307,...,0.192270,0.183969,0.397666,0.598897,0.427903,0.254003,0.196526,0.180164,0.285649,1.0
1,13.545805,0.799017,0.110258,409.500687,0.000266,0.016306,4.614077,4.617764,4.630562,1.990894,...,0.109459,0.113770,0.399140,0.683249,0.486938,0.218707,0.125231,0.137112,0.166869,1.0
2,23.779350,0.539249,0.063523,1198.639625,0.000201,0.014193,4.495674,4.470425,4.471587,2.211519,...,0.131142,0.190444,0.463232,0.632157,0.448661,0.233819,0.131819,0.119620,0.197390,1.0
3,16.856000,0.773274,0.081487,583.762113,0.000135,0.011633,4.486680,4.482031,4.499627,1.721655,...,0.069346,0.075007,0.346575,0.771139,0.471487,0.166829,0.084200,0.083803,0.100030,1.0
4,10.517836,0.451345,0.110380,212.400119,0.000769,0.027726,4.642310,4.638205,4.624247,2.485948,...,0.238023,0.197424,0.425098,0.540527,0.422710,0.245984,0.202996,0.200608,0.307281,1.0


In [41]:
# Utility function to evaluate model performance

def evaluate(y_train, y_test, train_pred, y_pred):
    train_accuracy = round(accuracy_score(y_train, train_pred), 3)
    test_accuracy = round(accuracy_score(y_test, y_pred), 3)
    precision = round(precision_score(y_test, y_pred), 3)
    recall = round(recall_score(y_test, y_pred), 3)
    f1 = round(f1_score(y_test, y_pred), 3)
    print('Train accuracy:', train_accuracy)
    print('Test accuracy:', test_accuracy)
    print('Precision:', precision)
    print('Recall:', recall)
    print('F1 Score', f1)
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    return train_accuracy, test_accuracy, precision, recall, f1

In [42]:
# Dataframe to store model evaluation metrics

columns = ['data_description', 'data_dimensions', 'model_used',
           'training_time(s)', 'Train_Accuracy', 'Test_Accuracy',
           'Precision', 'Recall', 'F1 Score']
eval_df = pd.DataFrame(columns=columns)

In [43]:
# Split the X and y Dataset into the Training set and Test set
from sklearn.model_selection import train_test_split

X = feature_df.drop(columns=['class']).values
y = feature_df['class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20,
                                                    random_state=0, stratify=y)

In [44]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# GridSearchCV

In [9]:
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

# Create a Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)

# Create a GridSearchCV object and fit it to the training data
grid_search = GridSearchCV(estimator=rf_classifier, param_grid=param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Print the best parameters and the corresponding accuracy score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy score: ", grid_search.best_score_)

# Evaluate the model on the training data
best_classifier = grid_search.best_estimator_
train_accuracy = best_classifier.score(X_train, y_train)
print("Accuracy on the test set: ", train_accuracy)

# Evaluate the model on the testing data
best_classifier = grid_search.best_estimator_
test_accuracy = best_classifier.score(X_test, y_test)
print("Accuracy on the test set: ", test_accuracy)

train_accuracy = best_classifier.score(X_train, y_train)
print("Accuracy on the train set: ", train_accuracy)

Best parameters found:  {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best accuracy score:  0.8371212121212122
Accuracy on the test set:  0.8354978354978355


In [12]:
from sklearn.svm import SVC

# Define the parameter grid to search over
param_grid = {
    'C': [0.1, 1.0, 10.0],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Create a Support Vector Classifier
svc_classifier = SVC(random_state=42)

# Create a GridSearchCV object and fit it to the training data
grid_search = GridSearchCV(estimator=svc_classifier, param_grid=param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Print the best parameters and the corresponding accuracy score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy score: ", grid_search.best_score_)

# Evaluate the model on the testing data
best_classifier = grid_search.best_estimator_
test_accuracy = best_classifier.score(X_test, y_test)
print("Accuracy on the test set: ", test_accuracy)

train_accuracy = best_classifier.score(X_train, y_train)
print("Accuracy on the train set: ", train_accuracy)

Best parameters found:  {'C': 10.0, 'gamma': 'scale', 'kernel': 'rbf'}
Best accuracy score:  0.8358225108225108
Accuracy on the test set:  0.8277056277056277
Accuracy on the train set:  0.8469696969696969


In [13]:
from sklearn.neighbors import KNeighborsClassifier

# Define the parameter grid to search over
param_grid = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

# Create a KNN Classifier
knn_classifier = KNeighborsClassifier()

# Create a GridSearchCV object and fit it to the training data
grid_search = GridSearchCV(estimator=knn_classifier, param_grid=param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Print the best parameters and the corresponding accuracy score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy score: ", grid_search.best_score_)

# Evaluate the model on the testing data
best_classifier = grid_search.best_estimator_
test_accuracy = best_classifier.score(X_test, y_test)
print("Accuracy on the test set: ", test_accuracy)

train_accuracy = best_classifier.score(X_train, y_train)
print("Accuracy on the train set: ", train_accuracy)

Best parameters found:  {'n_neighbors': 7, 'p': 1, 'weights': 'distance'}
Best accuracy score:  0.8122294372294373
Accuracy on the test set:  0.7991341991341991
Accuracy on the train set:  1.0


## Random Forest Classifier

In [45]:
from sklearn.ensemble import RandomForestClassifier

# Create an instance of Random Forest Classifier
forest = RandomForestClassifier(max_depth=None, min_samples_split=2, n_estimators=200, random_state=1, n_jobs=8)

train_start_time = time.time()

# Fit the model
forest.fit(X_train, y_train)

train_time = time.time() - train_start_time
print('Training time:', train_time, 's')

# Measure model performance
y_pred = forest.predict(X_test)
train_pred = forest.predict(X_train)

train_accuracy, test_accuracy, precision, recall, f1 = evaluate(y_train, y_test,
                                                                train_pred, y_pred)

# creating evaluation dataframe entry
eval_df.loc[len(eval_df)] = ['Original', feature_df.shape, 'Random Forest Classifier',
                             round(train_time, 3), train_accuracy, test_accuracy,
                             precision, recall, f1]

Training time: 1.9882838726043701 s
Train accuracy: 1.0
Test accuracy: 0.838
Precision: 0.827
Recall: 0.838
F1 Score 0.833
Confusion Matrix:
 [[1003  195]
 [ 180  932]]


## Support Vector Classifier

In [46]:
from sklearn.svm import SVC

# Training SVM model

train_start_time = time.time()

svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

train_time = time.time() - train_start_time
print('Training time:', train_time, 's')

y_pred = svm.predict(X_test)
train_pred = svm.predict(X_train)

train_accuracy, test_accuracy, precision, recall, f1 = evaluate(y_train, y_test,
                                                                train_pred, y_pred)

# creating evaluation dataframe entry
eval_df.loc[len(eval_df)] = ['Original', feature_df.shape, 'Support Vector Classifier',
                             round(train_time, 3), train_accuracy, test_accuracy,
                             precision, recall, f1]

Training time: 2.8966877460479736 s
Train accuracy: 0.807
Test accuracy: 0.811
Precision: 0.801
Recall: 0.808
F1 Score 0.804
Confusion Matrix:
 [[974 224]
 [213 899]]


## KNN Classifier

In [47]:
from sklearn.neighbors import KNeighborsClassifier

KNNclassifier = KNeighborsClassifier(n_neighbors=8 , metric='minkowski', p=2)

train_start_time = time.time()
KNNclassifier.fit(X_train, y_train)
train_time = time.time() - train_start_time
print('Training time:', train_time, 's')

y_pred = KNNclassifier.predict(X_test)
train_pred = KNNclassifier.predict(X_train)

train_accuracy, test_accuracy, precision, recall, f1 = evaluate(y_train, y_test,
                                                                train_pred, y_pred)

# creating evaluation dataframe entry
eval_df.loc[len(eval_df)] = ['Original', feature_df.shape, 'KNN Classifier',
                             round(train_time, 3), train_accuracy, test_accuracy,
                             precision, recall, f1]

Training time: 0.0024318695068359375 s
Train accuracy: 0.853
Test accuracy: 0.801
Precision: 0.792
Recall: 0.795
F1 Score 0.794
Confusion Matrix:
 [[966 232]
 [228 884]]


## Logistic Regression

In [48]:
from sklearn import linear_model

logr = linear_model.LogisticRegression(max_iter=1000)

train_start_time = time.time()

logr.fit(X_train, y_train)

train_time = time.time() - train_start_time
print('Training time:', train_time, 's')

y_pred = logr.predict(X_test)
train_pred = logr.predict(X_train)

train_accuracy, test_accuracy, precision, recall, f1 = evaluate(y_train, y_test,
                                                                train_pred, y_pred)

# creating evaluation dataframe entry
eval_df.loc[len(eval_df)] = ['Original', feature_df.shape, 'Logistic Regression',
                             round(train_time, 3), train_accuracy, test_accuracy,
                             precision, recall, f1]

Training time: 0.3877830505371094 s
Train accuracy: 0.804
Test accuracy: 0.814
Precision: 0.805
Recall: 0.81
F1 Score 0.808
Confusion Matrix:
 [[980 218]
 [211 901]]


In [49]:
eval_df

,data_description,data_dimensions,model_used,training_time(s),Train_Accuracy,Test_Accuracy,Precision,Recall,F1 Score
0,Original,"(11550, 49)",Random Forest Classifier,1.988,1.000,0.838,0.827,0.838,0.833
1,Original,"(11550, 49)",Support Vector Classifier,2.897,0.807,0.811,0.801,0.808,0.804
2,Original,"(11550, 49)",KNN Classifier,0.002,0.853,0.801,0.792,0.795,0.794
3,Original,"(11550, 49)",Logistic Regression,0.388,0.804,0.814,0.805,0.810,0.808


# Saving trained models
import pickle

pickle.dump(forest, open('trained_models/RandomForestClassifier.sav', 'wb'))
pickle.dump(svm, open('trained_models/SupportVectorClassifier.sav', 'wb'))
pickle.dump(KNNclassifier, open('trained_models/KNNClassifier.sav', 'wb'))
pickle.dump(logr, open('trained_models/LogisticRegression.sav', 'wb'))

In [17]:
# TEST ONE IMAGE

mean_std_df = pd.read_csv('mean_std_df.csv')

def my_standard_scaler(features):
    scaled_features = []
    for i, value in enumerate(features):
        mean = mean_std_df.iat[i, 1]
        std = mean_std_df.iat[i, 2]
        scaled_features.append((value - mean) / std)
    return np.array(scaled_features)

import cv2
import Utilities

testimg = cv2.imread('/Users/sinner/Desktop/waste_only_images_all/0067.jpg')
testimg = cv2.cvtColor(testimg, cv2.COLOR_BGR2RGB)

# # Feature Scaling
features = my_standard_scaler(Utilities.get_features(testimg))

predicted_class = forest.predict(np.reshape(features, (1,-1)))
# predicted_class = forest.predict(features)
print(predicted_class)

[1.]


In [18]:
# TEST ALL WINDOWS

import cv2
import Utilities
import os

src_folder_path = '/Users/sinner/Desktop/images_to_scan'
dest_folder_path = '/Users/sinner/Desktop/scanning_output'


imgCount = 0

# iterating in source folder
for filename in os.listdir(src_folder_path):

    if not filename.endswith('.jpg'):
        continue

    print(f"Scanning {filename}")

    img = cv2.imread(os.path.join(src_folder_path, filename))

    window_list = Utilities.get_windows(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 100, 100)

    print(f"no. of windows extracted: {len(window_list)}")
    for window in window_list:

        # Feature Scaling
        features = Utilities.get_features(window)
        features = my_standard_scaler(features)

        # predicted_class = loaded_model.predict(np.reshape(features, (1,-1)))
        predicted_class = forest.predict(np.reshape(features, (1,-1)))
        print(predicted_class)

        if predicted_class[0] == 1:
            imgCount += 1
            cv2.imwrite(os.path.join(dest_folder_path, f'{imgCount}.jpg'),
                        cv2.cvtColor(window, cv2.COLOR_RGB2BGR))

        print(imgCount)

Scanning 0001.jpg
no. of windows extracted: 527
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[0.]
0
[1.]
1
[1.]
2
[1.]
3
[1.]
4
[1.]
5
[1.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[0.]
6
[1.]
7
[1.]
8
[1.]
9
[1.]
10
[0.]
1